# Neural-Forecast Prediction Pipeline

Prediction/inference pipeline for the BTC forecasting system.

This notebook mirrors the feature engineering pipeline from training for prediction time, ensuring consistency between training and inference. The pipeline processes raw 1-minute BTC data, aggregates it to 15-minute intervals, applies feature engineering, loads a trained model, and generates predictions.

**Key Requirements:**
- Inference latency: <100ms for production predictions
- Feature consistency: Mirrors exact training pipeline
- NeuralForecast-native: Uses NF's built-in predict() method
- Validation gates: Ensures data quality and leakage prevention

## Imports

In [ ]:
#!/usr/bin/env python3

import pandas as pd
import numpy as np
from pathlib import Path
import sys
from neuralforecast import NeuralForecast

# Import data processing functions
from utils.io import (
    load_raw_1min_data,
    aggregate_1min_to_15min,
    regularize_to_grid_utc,
    make_nf_canonical,
    save_parquet,
    timestamped_path
)

# Import validation functions
from utils.validate import (
    assert_regular_grid,
    assert_utc_eob,
    assert_shifted,
    assert_no_forward_fill_y
)

## Configuration

In [ ]:
# Path configuration
DATA_PATH = "data/raw/btcusd_1-min_data.csv"
PROCESSED_PATH = "data/processed"
EXPERIMENT_PATH = lambda h: f"experiments/h{h}"
REPORTS_PATH = "reports"

# Configuration (was argparse)
RAW_DATA_PATH = "data/raw/btcusd_1-min_data.csv"  # Path to raw 1-minute CSV data
MODEL_PATH = None  # Path to trained model directory (required - set before execution)
HORIZON = 16  # Prediction horizon in 15-minute steps (1h, 2h, 4h, 8h = h4, h8, h16, h32)
OUTPUT_DIR = "predictions"  # Output directory for predictions
SAVE_PREDICTIONS = True  # Save predictions to disk

## Helper Functions

In [ ]:
def load_and_process_data(raw_data_path: str = "data/raw/btcusd_1-min_data.csv") -> pd.DataFrame:
    """
    Execute the complete data assembly path for prediction.
    
    This mirrors the training data pipeline to ensure consistency.
    
    Args:
        raw_data_path: Path to raw 1-minute CSV data
        
    Returns:
        Processed DataFrame ready for feature engineering
    """
    print("= Starting prediction data assembly...")
    
    # Step 1: Load raw 1-minute OHLCV data
    print("=🔄 Loading raw 1-minute data...")
    df_1min = load_raw_1min_data(raw_data_path)
    print(f"   Loaded {len(df_1min):,} 1-minute bars")
    
    # Step 2: Aggregate to 15-minute bars
    print("=📊 Aggregating 1-minute to 15-minute bars...")
    df_15min = aggregate_1min_to_15min(df_1min)
    print(f"   Aggregated to {len(df_15min):,} 15-minute bars")
    
    # Step 3: Regularize to complete UTC grid
    print("=🕐 Regularizing to UTC grid...")
    df_regularized = regularize_to_grid_utc(df_15min, "15min")
    print(f"   Regularized grid: {len(df_regularized):,} bars")
    
    # Step 4: Create NeuralForecast canonical format
    print("=📋 Creating NF canonical format...")
    df_canonical = make_nf_canonical(df_regularized, unique_id="BTC-USD")
    print(f"   Canonical format: {len(df_canonical):,} rows")
    
    # Validation gates
    print("✅ Running validation gates...")
    assert_regular_grid(df_canonical, "15min")
    print("    ✓ Regular grid validation passed")
    assert_utc_eob(df_canonical, "15min")
    print("    ✓ UTC EOB validation passed")
    
    print("=✅ Data assembly completed!")
    return df_canonical

In [ ]:
def integrate_features_for_prediction(nf_base: pd.DataFrame) -> tuple:
    """
    Integrate feature engineering pipeline for prediction.
    
    This mirrors the exact sequence from run_train.py to ensure
    consistent feature computation between training and inference.
    
    Args:
        nf_base: Canonical NF DataFrame from make_nf_canonical
        
    Returns:
        Tuple of (nf_df, hist_cols, futr_cols, stat_cols)
    """
    print("= Integrating feature engineering pipeline for prediction...")
    
    # Import feature engineering functions
    from features.registry import REGISTRY
    from features.builder import build_indicators, apply_mtf, postprocess_shift_and_prune, select_features
    
    # Exact sequence from Section 3.4 of docs/forecasting_sf_plan.md (mirrored from run_train.py)
    print("   =📈 Building base indicators...")
    base_feats = build_indicators(nf_base.rename(columns={"open":"open","high":"high","low":"low","close":"close","volume":"volume"}))
    
    print("   =🕐 Computing multi-timeframe features...")
    mtf_feats = apply_mtf(nf_base[["ds","open","high","low","close","volume"]])
    
    print("   =🔗 Merging feature sets...")
    exo_raw = base_feats.merge(mtf_feats, on="ds", how="left")
    
    print("   ⏩ Applying shift(1) and pruning...")
    exo = postprocess_shift_and_prune(exo_raw, rules={})
    
    print("   =🎯 Selecting features with hard cap...")
    hist_cols, futr_cols, stat_cols = select_features(exo, policy={})
    
    print("   =🔄 Merging features with canonical frame...")
    nf_df = nf_base.merge(exo, on="ds", how="left")
    
    # Validate leakage discipline before any NF call
    print("    🛡️ Validating leakage prevention with assert_shifted...")
    assert_shifted(nf_df, hist_cols)
    print("    ✓ Leakage validation passed!")
    
    # Print feature summary
    print(f"\n=📊 Feature Summary:")
    print(f"   Historical features: {len(hist_cols)}")
    print(f"   Future features: {len(futr_cols)}")
    print(f"   Static features: {len(stat_cols)}")
    print(f"   Total features: {len(hist_cols) + len(futr_cols) + len(stat_cols)} (cap: 256)")
    
    return nf_df, hist_cols, futr_cols, stat_cols

## Model Loading

In [ ]:
def load_trained_model(model_path: str, hist_cols: list, futr_cols: list, stat_cols: list):
    """
    Load a trained NeuralForecast model for prediction.
    
    Args:
        model_path: Path to saved model
        hist_cols: List of historical exogenous features
        futr_cols: List of future exogenous features
        stat_cols: List of static exogenous features
        
    Returns:
        Loaded NeuralForecast model
    """
    print(f"=🤖 Loading trained model from: {model_path}")
    
    # NeuralForecast.load() handles all model restoration
    # The feature lists ensure consistency with training
    nf = NeuralForecast.load(path=model_path)
    
    print("    ✓ Model loaded successfully")
    print(f"   Models: {[type(m).__name__ for m in nf.models]}")
    
    return nf

## Prediction Generation

In [ ]:
def generate_predictions(nf_model, df_predict: pd.DataFrame, horizon: int = 16):
    """
    Generate predictions using the loaded model.
    
    Args:
        nf_model: Loaded NeuralForecast model
        df_predict: DataFrame with features for prediction
        horizon: Prediction horizon in steps
        
    Returns:
        DataFrame with predictions
    """
    print(f"=🔮 Generating predictions for horizon={horizon}...")
    
    # Use NeuralForecast's native predict method
    predictions = nf_model.predict(df=df_predict, h=horizon)
    
    print(f"    ✓ Generated {len(predictions)} predictions")
    return predictions

## Execution

Main prediction pipeline execution. Make sure to set the `MODEL_PATH` variable in the configuration section above before running this cell.

**Performance Note:** This pipeline is designed to meet the <100ms inference latency requirement for production predictions.

In [ ]:
# Validate configuration
if MODEL_PATH is None:
    raise ValueError("MODEL_PATH must be set in the configuration section above")

try:
    # Step 1: Load and process data
    print("🚀 Starting prediction pipeline...\n")
    df_processed = load_and_process_data(RAW_DATA_PATH)
    
    # Step 2: Integrate feature engineering pipeline (mirrors training)
    nf_df, hist_cols, futr_cols, stat_cols = integrate_features_for_prediction(df_processed)
    
    # Step 3: Load trained model with feature lists
    nf_model = load_trained_model(
        MODEL_PATH, 
        hist_cols=hist_cols,
        futr_cols=futr_cols,
        stat_cols=stat_cols
    )
    
    # Step 4: Generate predictions
    predictions = generate_predictions(nf_model, nf_df, HORIZON)
    
    # Step 5: Save predictions if requested
    if SAVE_PREDICTIONS:
        output_path = timestamped_path(OUTPUT_DIR, "predictions", "parquet")
        save_parquet(predictions, output_path)
        print(f"=💾 Saved predictions to: {output_path}")
    
    # Show summary
    print("\n=📊 Prediction Summary:")
    print(f"   Total predictions: {len(predictions)}")
    print(f"   Horizon: {HORIZON} steps")
    if hasattr(predictions, 'index'):
        print(f"   Date range: {predictions.index.min()} to {predictions.index.max()}")
    
    # Show sample predictions
    print("\n=📋 Sample Predictions (first 5):")
    print(predictions.head())
    
except Exception as e:
    print(f"❌ Prediction pipeline failed: {e}")
    raise e

## Results Analysis

After running the prediction pipeline, you can analyze the results:

In [ ]:
# Display prediction results if available
try:
    if 'predictions' in locals():
        print("📈 Prediction Results:")
        print(f"Shape: {predictions.shape}")
        print(f"Columns: {list(predictions.columns)}")
        print(f"\nData types:")
        print(predictions.dtypes)
        print(f"\nBasic statistics:")
        print(predictions.describe())
    else:
        print("⚠️ No predictions available. Run the execution cell first.")
except NameError:
    print("⚠️ No predictions available. Run the execution cell first.")